# Matching volume

The volume information has not been importet. We have to retrieve it ex post

# Import and Data

In [18]:
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

# Configuration
folder_path = Path(r"C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues")
csv_path = "export-stutzmann-cmdf-t110-manuscript.csv"  # À remplacer par le chemin de votre CSV
output_csv = "..\\..\\out\\export-stutzmann-cmdf-t110-manuscript_vol.csv"  # Nom du fichier de sortie


# Process

In [21]:

# Charger le CSV
df = pd.read_csv(csv_path, sep='\t')

# Créer un dictionnaire pour mapper TEI_id -> liste de fichiers
tei_to_files = {}

# Parcourir tous les fichiers XML dans le dossier
for xml_file in folder_path.glob("*.xml"):
    print(xml_file)
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        # Gérer les namespaces TEI
        namespaces = {'tei': 'http://www.tei-c.org/ns/1.0'}
        
        # Chercher les éléments TEI avec xml:id
        # Essayer d'abord avec namespace
        tei_elements = root.findall(".//tei:TEI[@{http://www.w3.org/XML/1998/namespace}id]", namespaces)
        
        # Si pas trouvé avec namespace, essayer sans
        if not tei_elements:
            tei_elements = root.findall(".//*[@{http://www.w3.org/XML/1998/namespace}id]")
        
        for elem in tei_elements:
            xml_id = elem.get('{http://www.w3.org/XML/1998/namespace}id')
            if xml_id:
                if xml_id not in tei_to_files:
                    tei_to_files[xml_id] = []
                tei_to_files[xml_id].append(xml_file.name)
    
    except Exception as e:
        print(f"Erreur lors du traitement de {xml_file.name}: {e}")

# Enrichir le DataFrame
def get_files_for_id(tei_id):
    if pd.isna(tei_id):
        return ""
    files = tei_to_files.get(str(tei_id), [])
    return "|".join(files) if files else ""

df['fichier_source'] = df['TEI_id'].apply(get_files_for_id)

# Sauvegarder le résultat
df.to_csv(output_csv, sep='\t', index=False)

print(f"✓ Enrichissement terminé!")
print(f"✓ {len(tei_to_files)} TEI_id uniques trouvés dans les fichiers XML")
print(f"✓ Fichier sauvegardé: {output_csv}")

# Afficher quelques statistiques
matched = df['fichier_source'].str.len() > 0
print(f"✓ {matched.sum()} lignes enrichies sur {len(df)} au total")

C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues\modified_CMDF_1enr.xml
C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues\modified_CMDF_2enr.xml
C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues\modified_CMDF_3enr.xml
C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues\modified_CMDF_4enr.xml
C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues\modified_CMDF_5enr.xml
C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues\modified_CMDF_6enr.xml
C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues\modified_CMDF_7enr.xml
C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues\modified_CMDF_Cambraienr.xml
C:\Users\stutzmann\Github\CMDF\enhancingIndex\catalogues\modified_CMDF_Laon-Saint-Quentin-Soissonsenr.xml
✓ Enrichissement terminé!
✓ 9752 TEI_id uniques trouvés dans les fichiers XML
✓ Fichier sauvegardé: ..\..\out\export-stutzmann-cmdf-t110-manuscript_vol.csv
✓ 9533 lignes enrichies sur 10002 au total
